In [ ]:
import json
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_validate, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve
import xgboost as xgb

# Load data
alignment_matrix = np.load("./sample_hacking_output_20251003_163455/alignment_matrix_M.npy")
with open('./sample_hacking_output_20251003_163455/predictions.json', 'r') as file:
    data = json.load(file)
ground_truth = np.array(data["indicator_vector_true"])

# Define models with expanded set
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM (RBF)": SVC(kernel="rbf", probability=True),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "XGBoost": xgb.XGBClassifier(random_state=42, eval_metric='logloss')
}

# Parameter grids for grid search
param_grids = {
    "Logistic Regression": {
        'logisticregression__C': [0.1, 1, 10],
        'logisticregression__penalty': ['l2'],
        'logisticregression__solver': ['lbfgs']
    },
    "SVM (RBF)": {
        'svc__C': [0.1, 1, 10],
        'svc__gamma': ['scale', 'auto', 0.1, 0.01]
    },
    "Random Forest": {
        'randomforestclassifier__n_estimators': [100, 200, 300],
        'randomforestclassifier__max_depth': [None, 10, 20],
        'randomforestclassifier__min_samples_split': [2, 5]
    },
    "Gradient Boosting": {
        'gradientboostingclassifier__n_estimators': [100, 200],
        'gradientboostingclassifier__learning_rate': [0.01, 0.1],
        'gradientboostingclassifier__max_depth': [3, 5]
    },
    "XGBoost": {
        'xgbclassifier__n_estimators': [100, 200],
        'xgbclassifier__learning_rate': [0.01, 0.1],
        'xgbclassifier__max_depth': [3, 5]
    }
}

# Cross-validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Evaluate models with grid search
results = {}
best_estimators = {}
for name, model in models.items():
    print(f"\nTraining {name} with GridSearchCV...")
    pipe = make_pipeline(StandardScaler(), model)
    grid_search = GridSearchCV(pipe, param_grid=param_grids[name], cv=cv, scoring='f1', n_jobs=-1)
    grid_search.fit(alignment_matrix, ground_truth)
    
    # Store best model and scores
    best_estimators[name] = grid_search.best_estimator_
    scores = cross_validate(grid_search.best_estimator_, alignment_matrix, ground_truth, cv=cv,
                           scoring=["accuracy", "f1", "precision", "recall", "roc_auc"],
                           return_train_score=False)
    results[name] = {metric: (np.mean(scores[f"test_{metric}"]), np.std(scores[f"test_{metric}"]))
                     for metric in ["accuracy", "f1", "precision", "recall", "roc_auc"]}
    print(f"Best parameters for {name}: {grid_search.best_params_}")

# Print results
print("\nModel performance (mean ± std):")
for name, metrics in results.items():
    print(f"\n{name}:")
    for metric, (mean_val, std_val) in metrics.items():
        print(f"  {metric:10s}: {mean_val:.3f} ± {std_val:.3f}")

# Pick best model by F1 score
best_model_name = max(results, key=lambda k: results[k]["f1"][0])
print(f"\nBest model based on F1 score: {best_model_name}")

# Train/test split for final evaluation
X_train, X_test, y_train, y_test = train_test_split(alignment_matrix, ground_truth,
                                                    test_size=0.2, stratify=ground_truth,
                                                    random_state=42)

# Train best model
best_model = best_estimators[best_model_name]
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Class 0", "Class 1"], yticklabels=["Class 0", "Class 1"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title(f"Confusion Matrix - {best_model_name}")
plt.show()

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"{best_model_name} (AUC = {roc_auc_score(y_test, y_prob):.3f})")
plt.plot([0,1], [0,1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

# Feature importance for tree-based models
if best_model_name in ["Random Forest", "Gradient Boosting", "XGBoost"]:
    model_step = best_model.named_steps[best_model_name.lower().replace(" ", "")] if best_model_name != "XGBoost" else best_model.named_steps['xgbclassifier']
    importances = model_step.feature_importances_
    indices = np.argsort(importances)[::-1]
    plt.figure(figsize=(8,6))
    plt.bar(range(10), importances[indices[:10]], align="center")
    plt.xticks(range(10), [f"Feature {i}" for i in indices[:10]], rotation=45)
    plt.title(f"Top 10 Feature Importances - {best_model_name}")
    plt.xlabel("Feature")
    plt.ylabel("Importance")
    plt.tight_layout()
    plt.show()

# PCA Visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(StandardScaler().fit_transform(alignment_matrix))
plt.figure(figsize=(8,6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=ground_truth, cmap='viridis', alpha=0.6)
plt.title("PCA Visualization")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)")
plt.colorbar(scatter, label='Class')
plt.show()

# t-SNE Visualization
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(StandardScaler().fit_transform(alignment_matrix))
plt.figure(figsize=(8,6))
scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=ground_truth, cmap='viridis', alpha=0.6)
plt.title("t-SNE Visualization")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.colorbar(scatter, label='Class')
plt.show()


Training Logistic Regression with GridSearchCV...
Best parameters for Logistic Regression: {'logisticregression__C': 10, 'logisticregression__penalty': 'l2', 'logisticregression__solver': 'lbfgs'}

Training SVM (RBF) with GridSearchCV...
